**1. Setup**

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import loguniform

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

**2. Load Data & clean**

In [2]:
df = pd.read_csv("diabetes.csv")

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.shape

(768, 9)

**3. Quick checks**

In [5]:
# check missing values
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

**4. Split features & target**

In [6]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

In [7]:
X[:5]

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33


In [8]:
y[:5]

0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

**Model Training, Tuning, and Evaluation Flow**

Full data<br>
↓<br>
Train–test split<br>
↓<br>
GridSearchCV / RandomizedSearchCV<br>
↓<br>
CV Fold 1: fit scaler + model on train fold, validate<br>
↓<br>
CV Fold 2: fit scaler + model on train fold, validate<br>
↓<br>
CV Fold 3: ...<br>
↓<br>
Best hyperparameters selected<br>
↓<br>
Final evaluation on test set (once)

---

**Key**

❌ Cross-validation is NOT done on the full dataset<br>
❌ Test data is NOT used during hyperparameter tuning<br>
<br>
✅ Cross-validation happens only inside the training set<br>
✅ Test set is used only once, at the very end

**5. Train Test Split**

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [10]:
print("Dataset size:", X.shape)
print("Training data size", X_train.shape)
print("Test data size:", X_test.shape)

Dataset size: (768, 8)
Training data size (614, 8)
Test data size: (154, 8)


**6. Pipeline (scaler + model)**

In [11]:
pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]
)

**7. GridSearchCV (tries all combinations)**

In [12]:
param_grid = [
    {
        "model__kernel": ["linear"],
        "model__C": [0.1, 1, 10, 100]
    },
    {
        "model__kernel": ["rbf"],
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale", "auto", 0.01, 0.1,]
    },
    {
        "model__kernel": ["poly"],
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", "auto", 0.01, 0.1],
        "model__degree": [2, 3, 4]
    }
]

In [13]:
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1
)

In [14]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', SVC())]),
             n_jobs=-1,
             param_grid=[{'model__C': [0.1, 1, 10, 100],
                          'model__kernel': ['linear']},
                         {'model__C': [0.1, 1, 10, 100],
                          'model__gamma': ['scale', 'auto', 0.01, 0.1],
                          'model__kernel': ['rbf']},
                         {'model__C': [0.1, 1, 10], 'model__degree': [2, 3, 4],
                          'model__gamma': ['scale', 'auto', 0.01, 0.1],
                          'model__kernel': ['poly']}],
             return_train_score=True, scoring='accuracy')

In [15]:
print("GRID SEARCH CV - Results:")
print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

GRID SEARCH CV - Results:
Best params: {'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
Best CV accuracy: 0.7785419165667067


In [16]:
# evaluation on global test set (unseen data)
test_accuracy = round(grid.score(X_test, y_test)*100, 2)
print(f"Test Accuracy: {test_accuracy} %")

Test Accuracy: 75.32 %


**Analysing results**

In [17]:
grid.cv_results_

{'mean_fit_time': array([0.03668442, 0.05496535, 0.16081796, 0.50433474, 0.03951435,
        0.04310346, 0.02573366, 0.03428407, 0.03361621, 0.02299738,
        0.02115135, 0.0267695 , 0.03436613, 0.03137226, 0.04354682,
        0.04145756, 0.07589626, 0.07594628, 0.05997329, 0.10300746,
        0.0323813 , 0.02407041, 0.02473741, 0.0224606 , 0.03718915,
        0.03075166, 0.03026938, 0.01999326, 0.02638822, 0.02626667,
        0.02738109, 0.06613941, 0.02661152, 0.02528458, 0.02161469,
        0.03476315, 0.02396321, 0.02306952, 0.02847967, 0.04798827,
        0.04705958, 0.04083252, 0.02988086, 0.03272562, 0.05691648,
        0.09833183, 0.04878945, 0.08654881, 0.08498878, 0.09331012,
        0.02589445, 0.03820419, 0.06035438, 0.04753613, 0.03263106,
        0.0422502 ]),
 'std_fit_time': array([0.00959283, 0.02163246, 0.06943617, 0.09175315, 0.02031546,
        0.02277141, 0.00309482, 0.0220591 , 0.0170362 , 0.00164558,
        0.00081548, 0.00332916, 0.00309477, 0.00356252, 0.042

In [18]:
# gris searcg cv - all cv results

grid_results_df = pd.DataFrame(grid.cv_results_)

# select only the useful columns
grid_results_df = grid_results_df[["params", "mean_train_score", "mean_test_score", "rank_test_score"]].sort_values("rank_test_score")

grid_results_df

,params,mean_train_score,mean_test_score,rank_test_score
14,"{'model__C': 10, 'model__gamma': 0.01, 'model_...",0.800902,0.778542,1
0,"{'model__C': 0.1, 'model__kernel': 'linear'}",0.790718,0.778529,2
10,"{'model__C': 1, 'model__gamma': 0.01, 'model__...",0.789904,0.776929,3
2,"{'model__C': 10, 'model__kernel': 'linear'}",0.794791,0.775290,4
1,"{'model__C': 1, 'model__kernel': 'linear'}",0.795197,0.775290,4
3,"{'model__C': 100, 'model__kernel': 'linear'}",0.795198,0.773664,6
18,"{'model__C': 100, 'model__gamma': 0.01, 'model...",0.827775,0.762308,7
11,"{'model__C': 1, 'model__gamma': 0.1, 'model__k...",0.833473,0.758963,8
4,"{'model__C': 0.1, 'model__gamma': 'scale', 'mo...",0.779728,0.754085,9
5,"{'model__C': 0.1, 'model__gamma': 'auto', 'mod...",0.779728,0.754085,9


In [19]:
len(grid_results_df)

56

**8. Randomized Search CV (random combinations)**

In [20]:
param_dist = [
    # Linear
    {
        "model__kernel": ["linear"],
        "model__C": loguniform(0.01, 100)
    },
 
    # RBF
    {
        "model__kernel": ["rbf"],
        "model__C": loguniform(0.01, 1000),
        "model__gamma": loguniform(0.0001, 10)
    },

    # Poly
    {
        "model__kernel": ["poly"],
        "model__C": loguniform(0.01, 100),
        "model__gamma": loguniform(0.0001, 1),
    }
]

In [21]:
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    return_train_score=True,
    n_jobs=-1
)

In [22]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('model', SVC())]),
                   n_iter=20, n_jobs=-1,
                   param_distributions=[{'model__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000002ABB5B50F90>,
                                         'model__kernel': ['linear']},
                                        {'model__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000002ABB63BA...
                                         'model__gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000002ABB63B8490>,
                                         'model__kernel': ['rbf']},
                                        {'model__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000002ABB63B8A50>,
                                         'model__gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000002ABB63BA9D0>,
                                         'model__kernel': ['poly']}],
                   random_state=42, return_train_score=True,
                   scoring='accuracy')

In [23]:
print("Randomized SEARCH CV - Results:")
print("Best params:", random_search.best_params_)
print("Best CV accuracy:", random_search.best_score_)

Randomized SEARCH CV - Results:
Best params: {'model__C': 4.205156450913866, 'model__gamma': 0.01444525102276306, 'model__kernel': 'rbf'}
Best CV accuracy: 0.7850593096094894


In [24]:
# evaluation on global test set (unseen dataset)
test_accuracy = round(random_search.score(X_test, y_test)*100, 2)
print(f"Test accuracy: {test_accuracy} %")

Test accuracy: 74.03 %


In [25]:
# -----------------------------
# RandomizedSearchCV - All CV results
# -----------------------------
random_search_results_df = pd.DataFrame(random_search.cv_results_)

# Select only the most useful columns
random_search_results_df = random_search_results_df[
    [
        "params",
        "mean_train_score",
        "mean_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")

random_search_results_df

,params,mean_train_score,mean_test_score,rank_test_score
7,"{'model__C': 4.205156450913866, 'model__gamma'...",0.799275,0.785059,1
3,"{'model__C': 2.5378155082656657, 'model__kerne...",0.794383,0.775290,2
8,"{'model__C': 1.256315277393867, 'model__kernel...",0.793569,0.775290,2
12,"{'model__C': 0.015339162591163621, 'model__ker...",0.782573,0.773677,4
1,"{'model__C': 2.440060709081755, 'model__kernel...",0.793976,0.773664,5
6,"{'model__C': 2.950706670790534, 'model__kernel...",0.794383,0.773664,5
16,"{'model__C': 26.373339933815235, 'model__gamma...",0.824923,0.763934,7
17,"{'model__C': 2.754143921332031, 'model__gamma'...",0.868490,0.737892,8
10,"{'model__C': 0.6672367170464207, 'model__gamma...",0.809042,0.731361,9
4,"{'model__C': 0.012087541473056965, 'model__gam...",0.835916,0.729708,10


In [26]:
random_search_results_df[:1]["params"].values

array([{'model__C': 4.205156450913866, 'model__gamma': 0.01444525102276306, 'model__kernel': 'rbf'}],
      dtype=object)